In [9]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, mean_absolute_error, r2_score, mean_squared_error

PROJECT_ROOT = Path("..")   # 你的 notebook 在 notebooks/ 里
DATA_DIR = PROJECT_ROOT / "data"
ART_DIR = PROJECT_ROOT / "results" / "model_artifacts"

DATA_DIR, ART_DIR


(PosixPath('../data'), PosixPath('../results/model_artifacts'))

In [10]:
from pathlib import Path

def find_project_root(start: Path):
    cur = start.resolve()
    for _ in range(6):
        if (cur / "data" / "df_feat.parquet").exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError("Could not find project root containing data/df_feat.parquet")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
ART_DIR = PROJECT_ROOT / "results" / "model_artifacts"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_DIR exists:", ART_DIR.exists())
sorted([p.name for p in ART_DIR.glob("*")])[:10]


PROJECT_ROOT: /Users/wenxi/Desktop/TFM_25
ART_DIR exists: True


['clf_ed_xgb.joblib',
 'clf_ed_xgb.meta.json',
 'clf_highcost_rf.joblib',
 'clf_highcost_rf.meta.json',
 'clf_ip_rf.joblib',
 'clf_ip_rf.meta.json',
 'reg_log_totexpy2_xgb_es.booster.json',
 'reg_log_totexpy2_xgb_es.meta.json',
 'reg_log_totexpy2_xgb_es.preprocess.joblib']

In [11]:
df_feat = pd.read_parquet(DATA_DIR / "df_feat.parquet")
df_feat.shape, df_feat.columns[:10]


((7812, 141),
 Index(['DUID', 'PID', 'DUPERSID', 'PANEL', 'YEARIND', 'ALL5RDS', 'DIED',
        'INST', 'MILITARY', 'ENTRSRVY'],
       dtype='object'))

In [12]:
for c in ["LONGWT", "VARSTR", "VARPSU"]:
    print(c, c in df_feat.columns, df_feat[c].isna().sum() if c in df_feat.columns else None)


LONGWT True 0
VARSTR True 0
VARPSU True 0


In [13]:
import sys
sys.path.append(str(PROJECT_ROOT))
from src.models import split_train_val_test



#  saved 3 classification pipeline + meta（with threshold） sklearn 1.7.2

In [14]:
# HIGHCOST RF
clf_highcost = joblib.load(ART_DIR / "clf_highcost_rf.joblib")
meta_highcost = json.loads((ART_DIR / "clf_highcost_rf.meta.json").read_text())
t_highcost = meta_highcost["best_threshold"]

# ED XGB
clf_ed = joblib.load(ART_DIR / "clf_ed_xgb.joblib")
meta_ed = json.loads((ART_DIR / "clf_ed_xgb.meta.json").read_text())
t_ed = meta_ed["best_threshold"]

# IP RF
clf_ip = joblib.load(ART_DIR / "clf_ip_rf.joblib")
meta_ip = json.loads((ART_DIR / "clf_ip_rf.meta.json").read_text())
t_ip = meta_ip["best_threshold"]

t_highcost, t_ed, t_ip


/opt/anaconda3/envs/meps/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/meps/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/meps/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.7.2 when using version 1.8.0. Th

(0.5499999999999999, 0.2, 0.44999999999999996)

# saved regression model: booster + preprocess + best_iter

In [15]:
pre_reg = joblib.load(ART_DIR / "reg_log_totexpy2_xgb_es.preprocess.joblib")

booster = xgb.Booster()
booster.load_model(ART_DIR / "reg_log_totexpy2_xgb_es.booster.json")

meta_reg = json.loads((ART_DIR / "reg_log_totexpy2_xgb_es.meta.json").read_text())
best_iter = int(meta_reg["best_iteration"])

best_iter


1037

# classfication: weighted AUC/PR-AUC + weighted precision/recall

In [16]:
def weighted_clf_metrics(y_true, proba, w, threshold):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba).astype(float)
    w = np.asarray(w).astype(float)

    auc_w = roc_auc_score(y_true, proba, sample_weight=w)
    pr_w = average_precision_score(y_true, proba, sample_weight=w)

    pred = (proba >= threshold).astype(int)
    tp = w[(y_true==1) & (pred==1)].sum()
    fp = w[(y_true==0) & (pred==1)].sum()
    fn = w[(y_true==1) & (pred==0)].sum()

    precision_w = tp / (tp + fp + 1e-12)
    recall_w = tp / (tp + fn + 1e-12)

    return {"AUC_w": float(auc_w), "PR_AUC_w": float(pr_w),
            "precision_w": float(precision_w), "recall_w": float(recall_w)}


# regression：weighted RMSE/MAE/R²

In [17]:
def weighted_reg_metrics(y_true, y_pred, w):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    w = np.asarray(w).astype(float)

    rmse_w = np.sqrt(mean_squared_error(y_true, y_pred, sample_weight=w))
    mae_w = mean_absolute_error(y_true, y_pred, sample_weight=w)
    r2_w = r2_score(y_true, y_pred, sample_weight=w)
    return {"RMSE_w": float(rmse_w), "MAE_w": float(mae_w), "R2_w": float(r2_w)}


# define “split + weight” helper

In [18]:
def split_with_weights(df, target, feature_cols, *, random_state=42, stratify=False,
                       w_col="LONGWT", str_col="VARSTR", psu_col="VARPSU"):
    tmp = df[feature_cols + [target, w_col, str_col, psu_col]].dropna()

    X = tmp[feature_cols].copy()
    y = tmp[target].copy()

    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        X, y, random_state=random_state, stratify=stratify
    )

    w_test = tmp.loc[X_test.index, w_col].astype(float).values
    return X_test, y_test, w_test


# weighted evaluation（on test ）

**HIGHCOST_Y2**

In [19]:
cols_hc = list(clf_highcost.feature_names_in_)
X_test, y_test, w_test = split_with_weights(df_feat, "HIGHCOST_Y2", cols_hc, random_state=42, stratify=True)

proba = clf_highcost.predict_proba(X_test[cols_hc])[:, 1]
hc_w = weighted_clf_metrics(y_test.values, proba, w_test, threshold=t_highcost)
hc_w


AttributeError: 'SimpleImputer' object has no attribute '_fill_dtype'

In [20]:
import json
from pathlib import Path

meta = json.loads((ART_DIR / "clf_highcost_rf.meta.json").read_text())
meta.get("sklearn_version"), meta


('1.7.2',
 {'target': 'HIGHCOST_Y2',
  'model': 'RandomForest',
  'best_threshold': 0.5499999999999999,
  'test_metrics': {'AUC': 0.8676443788384087,
   'PR_AUC': 0.45889778426594363,
   'F1_at_best_t': 0.5360230547550432},
  'saved_at': '2025-12-26T19:43:45.196702',
  'sklearn_version': '1.7.2',
  'xgboost_version': '3.1.2'})

In [21]:
import sklearn
sklearn.__version__


'1.8.0'

In [22]:
import sys
sys.path.append(str(PROJECT_ROOT))
from src.models import split_train_val_test
from src.preprocessing import make_preprocess


ImportError: cannot import name 'make_preprocess' from 'src.preprocessing' (/Users/wenxi/Desktop/TFM_25/notebooks/../src/preprocessing.py)

In [23]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import roc_auc_score, average_precision_score


In [24]:
PROJECT_ROOT = Path("..").resolve()   # notebook 在 notebooks/ 里
DATA_DIR = PROJECT_ROOT / "data"

df_feat = pd.read_parquet(DATA_DIR / "df_feat.parquet")
df_feat.shape


(7812, 141)

In [25]:
import sys
sys.path.append(str(PROJECT_ROOT))

from src.models import split_train_val_test
from src.models import make_preprocess   # 你之前 models.py 里有 make_preprocess


In [26]:
W_COL = "LONGWT"
STR_COL = "VARSTR"
PSU_COL = "VARPSU"

df_feat[[W_COL, STR_COL, PSU_COL]].isna().sum()


LONGWT    0
VARSTR    0
VARPSU    0
dtype: int64

In [6]:
cat_cols = [
    "RACE_ETH",
    "REGIONY1_CAT",
    "EDU_GROUP",
    "POVCATY1_CAT",
    "FAMSIZE_Y1_GRP",
    "INS_TYPE_Y1",
]

In [7]:
#Numeric (use engineered columns, not raw)

num_cols = [
    # demographics / SES
    "AGE",
    "SEX_BIN",
    "LOG_FAMINCY1",
    "FAMSIZE_Y1",

   

    # employment
    "WORKED_Y1",
    "ANY_UNEMP_COMP_Y1",
    "LOG_UNEMP_COMP_Y1",
    "EMP_INFO_R12",
    "EMP_ATTACHED_ANY_R12_FILL0",  # model-friendly version

    # health status baseline
    "RTHLTH1_FAIRPOOR",
    "MNHLTH1_FAIRPOOR",

    # chronic conditions baseline
    "HIBPDXY1_BIN",
    "CHDDXY1_BIN",
    "STRKDXY1_BIN",
    "CHOLDXY1_BIN",
    "ASTHDXY1_BIN",
    "DIABDXY1_M18_BIN",
    # "MULTIMORBIDITY_Y1",   # optional (can remove if you keep all *_BIN)

    # baseline utilisation/cost
    "LOG_TOTEXPY1",
    "ANY_ED_Y1",
    "ANY_IP_Y1",
]

In [8]:
FEATURES = num_cols + cat_cols


In [29]:
def weighted_clf_metrics(y_true, proba, w, threshold):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba).astype(float)
    w = np.asarray(w).astype(float)

    auc_w = roc_auc_score(y_true, proba, sample_weight=w)
    pr_w = average_precision_score(y_true, proba, sample_weight=w)

    pred = (proba >= threshold).astype(int)
    tp = w[(y_true==1) & (pred==1)].sum()
    fp = w[(y_true==0) & (pred==1)].sum()
    fn = w[(y_true==1) & (pred==0)].sum()

    precision_w = tp / (tp + fp + 1e-12)
    recall_w = tp / (tp + fn + 1e-12)

    return {"AUC_w": float(auc_w), "PR_AUC_w": float(pr_w),
            "precision_w": float(precision_w), "recall_w": float(recall_w)}


In [30]:
def build_split_with_weights(df, target, feature_cols, *, random_state=42, stratify=False):
    tmp = df[feature_cols + [target, W_COL]].dropna()
    X = tmp[feature_cols].copy()
    y = tmp[target].copy()

    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        X, y, random_state=random_state, stratify=stratify
    )
    w_test = tmp.loc[X_test.index, W_COL].astype(float).values
    return X_train, X_val, X_test, y_train, y_val, y_test, w_test


In [40]:
RANDOM_SEED = 42

# preprocessing（树模型不需要scale）
pre_tree = make_preprocess(num_cols,cat_cols, scale_numeric=False)

# ---- HIGHCOST RF ----
X_tr, X_va, X_te, y_tr, y_va, y_te, w_te = build_split_with_weights(
    df_feat, "HIGHCOST_Y2", FEATURES, random_state=RANDOM_SEED, stratify=True
)
rf_hc = RandomForestClassifier(
    n_estimators=800, max_depth=16, min_samples_leaf=3,
    random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample"
)
pipe_hc = Pipeline([("preprocess", pre_tree), ("model", rf_hc)])
pipe_hc.fit(X_tr, y_tr)
proba_hc = pipe_hc.predict_proba(X_te)[:, 1]

t_hc = 0.55
hc_w = weighted_clf_metrics(y_te.values, proba_hc, w_te, threshold=t_hc)
hc_w


{'AUC_w': 0.849763335535662,
 'PR_AUC_w': 0.39147541902771293,
 'precision_w': 0.4964422847105858,
 'recall_w': 0.38825690685445563}

In [41]:
# ---- ANY_ED XGB ----
X_tr, X_va, X_te, y_tr, y_va, y_te, w_te = build_split_with_weights(
    df_feat, "ANY_ED_Y2", FEATURES, random_state=RANDOM_SEED, stratify=True
)

xgb_ed = XGBClassifier(
    n_estimators=800, max_depth=3, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9,
    random_state=RANDOM_SEED, n_jobs=-1, tree_method="hist",
    eval_metric="logloss"
)
pipe_ed = Pipeline([("preprocess", pre_tree), ("model", xgb_ed)])
pipe_ed.fit(X_tr, y_tr)
proba_ed = pipe_ed.predict_proba(X_te)[:, 1]

t_ed = 0.20
ed_w = weighted_clf_metrics(y_te.values, proba_ed, w_te, threshold=t_ed)
ed_w


{'AUC_w': 0.7000538014536916,
 'PR_AUC_w': 0.2868167063671123,
 'precision_w': 0.2677941143346467,
 'recall_w': 0.4588862270083041}

In [42]:
# ---- ANY_IP RF ----
X_tr, X_va, X_te, y_tr, y_va, y_te, w_te = build_split_with_weights(
    df_feat, "ANY_IP_Y2", FEATURES, random_state=RANDOM_SEED, stratify=True
)

rf_ip = RandomForestClassifier(
    n_estimators=800, max_depth=16, min_samples_leaf=3,
    random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample"
)
pipe_ip = Pipeline([("preprocess", pre_tree), ("model", rf_ip)])
pipe_ip.fit(X_tr, y_tr)
proba_ip = pipe_ip.predict_proba(X_te)[:, 1]

t_ip = 0.40
ip_w = weighted_clf_metrics(y_te.values, proba_ip, w_te, threshold=t_ip)
ip_w


{'AUC_w': 0.7476741628670012,
 'PR_AUC_w': 0.19218476856949024,
 'precision_w': 0.2400953070219936,
 'recall_w': 0.23567980744083095}

Weighted evaluation using LONGWT produced slightly lower AUC/PR-AUC compared with the unweighted test evaluation, while precision–recall trade-offs shifted depending on the fixed operating threshold. Overall conclusions were unchanged: high-cost prediction remains the strongest task, ED is moderate, and inpatient admission is the most challenging due to rarity.

## error analysis

In [21]:
from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "data" / "df_feat.parquet").exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError("Could not find project root containing data/df_feat.parquet")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "df_feat.parquet"
ART_DIR = PROJECT_ROOT / "results" / "model_artifacts_sklearn 1.8.0"

df_feat = pd.read_parquet(DATA_PATH)
PROJECT_ROOT, ART_DIR.exists(), df_feat.shape


(PosixPath('/Users/wenxi/Desktop/TFM_25'), True, (7812, 141))

#  saved 3 classification pipeline + meta（with threshold） sklearn 1.8.0

In [22]:
# classification
clf_hc = joblib.load(ART_DIR / "clf_highcost_rf.joblib")
meta_hc = json.loads((ART_DIR / "clf_highcost_rf.meta.json").read_text())
t_hc = float(meta_hc["best_threshold"])
cols_hc = meta_hc["feature_cols"]

clf_ed = joblib.load(ART_DIR / "clf_ed_xgb.joblib")
meta_ed = json.loads((ART_DIR / "clf_ed_xgb.meta.json").read_text())
t_ed = float(meta_ed["best_threshold"])
cols_ed = meta_ed["feature_cols"]

clf_ip = joblib.load(ART_DIR / "clf_ip_rf.joblib")
meta_ip = json.loads((ART_DIR / "clf_ip_rf.meta.json").read_text())
t_ip = float(meta_ip["best_threshold"])
cols_ip = meta_ip["feature_cols"]

# regression (booster)
pre_reg = joblib.load(ART_DIR / "reg_log_totexpy2_xgb_es.preprocess.joblib")
booster = xgb.Booster()
booster.load_model(ART_DIR / "reg_log_totexpy2_xgb_es.booster.json")
meta_reg = json.loads((ART_DIR / "reg_log_totexpy2_xgb_es.meta.json").read_text())
best_iter = int(meta_reg["best_iteration"])
cols_reg = meta_reg["feature_cols"]

t_hc, t_ed, t_ip, best_iter


(0.49999999999999994, 0.15, 0.39999999999999997, 605)

主文（Results & Model selection）

以“模型家族层面”做结论：RF / XGB / RF / XGB-ES

用旧版本产生的数字作为主结果（你已经写了）

Error analysis / Appendix

用新 artifacts 做错误分析和加权敏感性分析

强调它是robustness / qualitative pattern：例如 FP/FN 在中等风险段、回归高费用误差更大等
这些结论对小幅数值变化不敏感。

4) 你要不要担心评审问“为什么不全用同一套模型”？

为了避免这种问题，你可以做一个很简单的“对齐”：

模型类型保持一致（仍然是 RF/XGB/RF/XGB-ES）

阈值仍用主文的阈值（0.55/0.20/0.40 或你定的）

error analysis 只解释“错误分布/误差结构”，不拿它重新做模型对比

这样评审很难挑毛病。

A) Classification error analysis（TP/FP/FN/TN + FP vs FN 对比）

In [19]:
from pathlib import Path
import sys

# notebook 在 notebooks/ 里 → project root 是上一级
PROJECT_ROOT = Path("..").resolve()

# 把项目根目录加入 Python 搜索路径
sys.path.append(str(PROJECT_ROOT))

# 验证
print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())


PROJECT_ROOT: /Users/wenxi/Desktop/TFM_25
src exists: True


In [20]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from src.models import split_train_val_test

def classification_error_table(
    df_feat: pd.DataFrame,
    target: str,
    feature_cols: list[str],
    fitted_model,                 # sklearn Pipeline with predict_proba
    threshold: float,
    *,
    random_state: int = 42,
    stratify: bool = True
):
    """
    Build an error table on the test split:
    - proba, predicted label, and error type (TP/FP/FN/TN)
    Returns:
      df_err (test rows with diagnostics), cm_dict
    """
    tmp = df_feat[feature_cols + [target]].dropna()
    X = tmp[feature_cols].copy()
    y = tmp[target].astype(int)

    X_tr, X_va, X_te, y_tr, y_va, y_te = split_train_val_test(
        X, y, random_state=random_state, stratify=stratify
    )

    proba = fitted_model.predict_proba(X_te)[:, 1]
    y_pred = (proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    cm_dict = {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

    df_err = X_te.copy()
    df_err["y_true"] = y_te.values
    df_err["proba"] = proba
    df_err["y_pred"] = y_pred

    def _etype(r):
        if r.y_true == 1 and r.y_pred == 1: return "TP"
        if r.y_true == 0 and r.y_pred == 1: return "FP"
        if r.y_true == 1 and r.y_pred == 0: return "FN"
        return "TN"

    df_err["error_type"] = df_err.apply(_etype, axis=1)
    return df_err, cm_dict


def compare_fp_fn(
    df_err: pd.DataFrame,
    *,
    num_cols: list[str] | None = None,
    cat_cols: list[str] | None = None,
    topn_cat: int = 6
):
    """
    Compare FP vs FN on selected features.
    Returns:
      - numeric_means: FP mean vs FN mean
      - cat_distributions: dict of value distributions for FP and FN
    """
    num_cols = num_cols or []
    cat_cols = cat_cols or []

    fp = df_err[df_err["error_type"] == "FP"]
    fn = df_err[df_err["error_type"] == "FN"]

    out = {}

    if num_cols:
        numeric_means = pd.DataFrame({
            "FP_mean": fp[num_cols].mean(numeric_only=True),
            "FN_mean": fn[num_cols].mean(numeric_only=True),
        })
        numeric_means["diff_FP_minus_FN"] = numeric_means["FP_mean"] - numeric_means["FN_mean"]
        out["numeric_means"] = numeric_means.sort_values("diff_FP_minus_FN", ascending=False)

    if cat_cols:
        cat_tables = {}
        for c in cat_cols:
            fp_dist = fp[c].value_counts(normalize=True).head(topn_cat)
            fn_dist = fn[c].value_counts(normalize=True).head(topn_cat)
            cat_tables[c] = pd.DataFrame({"FP_pct": fp_dist, "FN_pct": fn_dist}).fillna(0)
        out["cat_distributions"] = cat_tables

    return out


# ===== Example: ANY_IP_Y2 (recommended for error analysis because hardest) =====
TARGET = "ANY_IP_Y2"
FEATURE_COLS = FEATURES          # 你论文最终特征：num_cols + cat_cols
THRESHOLD = 0.40                 # 用你主文的 operating point

df_ip_err, cm_ip = classification_error_table(
    df_feat, TARGET, FEATURE_COLS,
    fitted_model=clf_ip,         # 你重训的 pipeline（或加载的 clf_ip）
    threshold=THRESHOLD,
    random_state=42,
    stratify=True
)

cm_ip, df_ip_err["error_type"].value_counts()


AttributeError: 'SimpleImputer' object has no attribute '_fill_dtype'